In [1]:
!pip install -q PyMuPDF


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -q faiss-cpu sentence-transformers PyMuPDF


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import pickle
from typing import List
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

c:\Users\Abdur Rehman\Desktop\Homework-Week3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import fitz  # PyMuPDF

def load_pdf_text(path: str) -> str:
    """
    Extracts text from a PDF file.
    """
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

In [5]:
pdf_path = "./salahuddin_ayubi.pdf"
text = load_pdf_text(pdf_path)

In [6]:
def preprocess_text(text: str) -> str:
    return ' '.join(text.split())

cleaned_text = preprocess_text(text)

In [7]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50):
    """
    Break text into overlapping chunks.
    """
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(cleaned_text)

In [8]:
from retriever import Retriever

retriever = Retriever(chunk_size=500)
retriever.add_documents([cleaned_text])

Adding 1 documents to the retriever. 500


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.44it/s]


In [9]:
results = retriever.query("Who was Salahuddin Ayubi and what is he known for?", top_k=3)

for i, (chunk, score) in enumerate(results):
    print(f"\n--- Result {i+1} [score: {score:.4f}] ---\n{chunk}")


--- Result 1 [score: 0.6984] ---
Salahuddin Ayubi, also known as Saladin in the Western world, was a prominent Muslim leader and military commander during the 12th century. He was born in Tikrit in 1137 and became the first Sultan of Egypt and Syria, founding the Ayyubid dynasty. Salahuddin is best known for his leadership during the Crusades, especially for recapturing Jerusalem from the Crusaders in 1187 after the Battle of Hattin. He was admired by both Muslims and Christians for his chivalry, leadership, and sense of justice. Despite being a formidable military commander, Salahuddin was known for treating his enemies with honor and compassion. His conduct during the siege and eventual surrender of Jerusalem won him respect across religious lines. Salahuddin's reign marked a turning point in Islamic history, revitalizing the Muslim world and unifying various factions to resist the Crusaders. He died in Damascus in 1193, leaving behind a legacy of unity, courage, and leadership.

--

In [10]:
def test_retriever_on_known_document():
    # Known document
    doc = """Salahuddin Ayubi, also known as Saladin, was a Muslim leader in the 12th century.
    He recaptured Jerusalem from the Crusaders in 1187. He was known for justice and honor."""
    
    # Known query
    query = "Who recaptured Jerusalem from the Crusaders?"

    retriever = Retriever(chunk_size=40)
    retriever.add_documents([doc])
    results = retriever.query(query, top_k=1)

    top_result = results[0][0].lower()

    assert "jerusalem" in top_result and "crusaders" in top_result, \
        "Test failed: top result does not mention expected content"

    print("✅ Test passed: Retriever returned expected chunk.")

# Run test
test_retriever_on_known_document()


Adding 1 documents to the retriever. 40


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.39it/s]

✅ Test passed: Retriever returned expected chunk.


In [11]:
# Example usage of the retriever with a known document
retriever = Retriever(chunk_size=100)
retriever.add_documents(["Salahuddin Ayubi recaptured Jerusalem."])
results = retriever.query("Who took back Jerusalem?")
print(results[0])

Adding 1 documents to the retriever. 100


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.87it/s]

('Salahuddin Ayubi recaptured Jerusalem.', 0.6978177428245544)
